# Workshop - Data Pipeline in Practice (GR5069)

###### Read in Dataset

In [0]:
from pyspark.sql.functions import datediff, current_date, avg,col 
from pyspark.sql.types import IntegerType

In [0]:
# dbutils.fs.ls('s3://columbia-gr5069-main/raw/')

In [0]:
df_pitstops = spark.read.csv('/Volumes/gr5069/raw/f1_data/pit_stops.csv', header=True)

In [0]:
display(df_pitstops)

In [0]:
display(df_pitstops.groupby('driverId').count())

In [0]:
display(df_pitstops.filter(col('duration').isNull()))

In [0]:
df_laptimes = spark.read.csv('/Volumes/gr5069/raw/f1_data/lap_times.csv', header=True)

In [0]:
display(df_laptimes)

In [0]:
df_driver = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv', header=True)
df_driver.count()

In [0]:
display(df_driver)

In [0]:
display(df_driver.groupby('code').count())

###### Transform Data

In [0]:
df_driver = df_driver.withColumn("age", datediff(current_date(),df_driver.dob)/365)

In [0]:
display(df_driver)
#

In [0]:
df_driver = df_driver.withColumn('age', df_driver['age'].cast(IntegerType()))

In [0]:
display(df_driver)

In [0]:
df_lap_drivers = df_driver.select('driverId','driverRef', 'forename', 'surname', 'nationality', 'age').join(df_laptimes, on=['driverId'])

In [0]:
display(df_lap_drivers)

###### Aggregate by Age

In [0]:
df_lap_drivers = df_lap_drivers.groupby('age').agg(avg('milliseconds'))

In [0]:
display(df_lap_drivers)

###### Storing Data in S3

In [0]:
df_lap_drivers.write.csv('/Volumes/gr5069/yy3462/inclass/lap_times_output.csv')